# NODDI Fitting on Lab Data (Subsampled Protocols)

**Purpose:** Fit NODDI model to 14 single-shell subsampled lab DTI protocols using AMICO  
**Data:** TEST03092026, 701 volumes total  
**Protocols:** 5 × b1000, 5 × b2000, 4 × b3000 (14 total single-shell protocols)  
**Output:** NDI, ODI, FWF maps for each protocol

---

## 1. Setup and Configuration

In [ ]:
#!/usr/bin/env python
"""
NODDI fitting using AMICO on lab subsampled protocols (14 single-shell)
Author: Raiyun Kabir
"""

import os
import sys
import numpy as np
import nibabel as nib
import amico
from pathlib import Path
import time
import shutil

print("Package versions:")
print(f"  NumPy:   {np.__version__}")
print(f"  NiBabel: {nib.__version__}")
print(f"  AMICO:   {amico.__version__}")

## 2. Configuration Parameters

Define protocol configurations and file paths.

In [ ]:
# ============================================================
# CONFIGURATION - MODIFY THESE
# ============================================================

PREPROC_DIR = "/gpfs/fs2/scratch/rkabir5/StarterCodes_Data/new_data/afaiyaz-20260313_190724/TEST03092026/preproc"
OUTPUT_ROOT = "/gpfs/fs2/scratch/rkabir5/StarterCodes_Data/new_data/afaiyaz-20260313_190724/TEST03092026/noddi_results"

# Input filenames (in PREPROC_DIR)
DWI_FILE   = "eddy_unwarped_images.nii.gz"
BVALS_FILE = "bvals"
BVECS_FILE = "eddy_unwarped_images.eddy_rotated_bvecs"
MASK_FILE  = "hifi_nodif_brain_mask.nii.gz"

# AMICO parameters
B0_THRESHOLD = 50
NUM_THREADS  = 8

# ============================================================
# 14 single-shell protocols — contiguous blocks from acquisition order
# ============================================================
PROTOCOLS = {
    'b1000_n20': {'shell': 1000, 'ndir': 20, 'start_vol':   0, 'n_vols': 24},
    'b1000_n30': {'shell': 1000, 'ndir': 30, 'start_vol':  24, 'n_vols': 34},
    'b1000_n45': {'shell': 1000, 'ndir': 45, 'start_vol':  58, 'n_vols': 49},
    'b1000_n60': {'shell': 1000, 'ndir': 60, 'start_vol': 107, 'n_vols': 64},
    'b1000_n90': {'shell': 1000, 'ndir': 90, 'start_vol': 171, 'n_vols': 94},
    'b2000_n20': {'shell': 2000, 'ndir': 20, 'start_vol': 265, 'n_vols': 24},
    'b2000_n30': {'shell': 2000, 'ndir': 30, 'start_vol': 289, 'n_vols': 34},
    'b2000_n45': {'shell': 2000, 'ndir': 45, 'start_vol': 323, 'n_vols': 49},
    'b2000_n60': {'shell': 2000, 'ndir': 60, 'start_vol': 372, 'n_vols': 64},
    'b2000_n90': {'shell': 2000, 'ndir': 90, 'start_vol': 436, 'n_vols': 94},
    'b3000_n20': {'shell': 3000, 'ndir': 20, 'start_vol': 530, 'n_vols': 24},
    'b3000_n30': {'shell': 3000, 'ndir': 30, 'start_vol': 554, 'n_vols': 34},
    'b3000_n45': {'shell': 3000, 'ndir': 45, 'start_vol': 588, 'n_vols': 49},
    'b3000_n60': {'shell': 3000, 'ndir': 60, 'start_vol': 637, 'n_vols': 64},
}

# Optionally run a single protocol (useful for testing or array jobs)
PROTOCOL_NAME = os.environ.get('PROTOCOL', None)

if PROTOCOL_NAME and PROTOCOL_NAME in PROTOCOLS:
    protocols_to_run = {PROTOCOL_NAME: PROTOCOLS[PROTOCOL_NAME]}
    print(f"Processing single protocol: {PROTOCOL_NAME}")
else:
    protocols_to_run = PROTOCOLS
    if PROTOCOL_NAME:
        print(f"Warning: PROTOCOL={PROTOCOL_NAME} not found. Processing all protocols.")
    else:
        print(f"Processing all {len(PROTOCOLS)} protocols")

# ============================================================
# Helper functions (defined once outside loop)
# ============================================================

def round_to_shell(bval):
    """Round b-value to nearest standard shell."""
    if bval < 50:
        return 0
    elif bval < 1500:
        return 1000
    elif bval < 2500:
        return 2000
    else:
        return 3000

def create_amico_scheme(bval_file, bvec_file, output_file):
    """Convert FSL bvals/bvecs to AMICO scheme format."""
    bvals = np.loadtxt(bval_file)
    bvecs = np.loadtxt(bvec_file)
    if bvecs.shape[0] == 3:
        bvecs = bvecs.T   # (3, N) → (N, 3)
    scheme = np.column_stack([bvecs, bvals])
    with open(output_file, 'w') as f:
        f.write("VERSION: BVECTOR\n")
        np.savetxt(f, scheme, fmt='%.6f')
    return scheme

# Initialize AMICO once (expensive — do NOT put inside loop)
print("\nInitializing AMICO framework...")
amico.core.setup()
print("✓ AMICO initialized\n")

JOB_ID   = os.environ.get('SLURM_JOB_ID', 'local')
TEMP_DIR = os.path.join(PREPROC_DIR, f"temp_noddi_sub_{JOB_ID}")

print(f"Configuration:")
print(f"  Job ID:           {JOB_ID}")
print(f"  PREPROC_DIR:      {PREPROC_DIR}")
print(f"  Temp dir:         {TEMP_DIR}")
print(f"  Output root:      {OUTPUT_ROOT}")
print(f"  Protocols to run: {list(protocols_to_run.keys())}")
print(f"  Threads:          {NUM_THREADS}")

## 3. Verify Source Files

Check that the full DWI, mask, bvals, and bvecs exist, then load the full DWI into memory once.
All 14 protocol extractions reuse the same in-memory array.

In [ ]:
FULL_DWI       = os.path.join(PREPROC_DIR, DWI_FILE)
FULL_BVALS     = os.path.join(PREPROC_DIR, BVALS_FILE)
FULL_BVECS     = os.path.join(PREPROC_DIR, BVECS_FILE)
MASK_FILE_PATH = os.path.join(PREPROC_DIR, MASK_FILE)

print("Checking source files...\n")
for label, fpath in [("Full DWI",   FULL_DWI),
                     ("Brain mask", MASK_FILE_PATH),
                     ("b-values",   FULL_BVALS),
                     ("b-vectors",  FULL_BVECS)]:
    status = "✓" if os.path.exists(fpath) else "✗ MISSING"
    print(f"  {status}  {label}: {fpath}")

if not os.path.exists(FULL_DWI):
    raise FileNotFoundError(f"Full DWI not found: {FULL_DWI}")
if not os.path.exists(MASK_FILE_PATH):
    raise FileNotFoundError(f"Brain mask not found: {MASK_FILE_PATH}")

print(f"\n✓ All files verified — ready to process {len(protocols_to_run)} protocols")

# ============================================================
# Load full DWI ONCE — all 14 extractions reuse this array
# ============================================================

print(f"\nLoading full DWI (this may take ~60s)...")
t0 = time.time()
full_img      = nib.load(FULL_DWI)
full_dwi_data = full_img.get_fdata()   # shape: (172, 172, 96, 701)
full_affine   = full_img.affine
full_header   = full_img.header
print(f"  ✓ Full DWI loaded: shape={full_dwi_data.shape}, {(time.time()-t0):.1f}s")

# Load full bvals/bvecs once
full_bvals = np.loadtxt(FULL_BVALS)    # (701,)
full_bvecs = np.loadtxt(FULL_BVECS)   # (3, 701) FSL convention

# ============================================================
# Main loop — one NODDI fit per protocol
# ============================================================

all_results = {}

for protocol_name, info in protocols_to_run.items():
    shell      = info['shell']
    ndir       = info['ndir']
    start_vol  = info['start_vol']
    n_vols     = info['n_vols']

    print("\n" + "="*70)
    print(f"PROTOCOL: {protocol_name}  (shell={shell}, dirs={ndir})")
    print("="*70)

    # ---- Paths -------------------------------------------------------
    proto_temp_dir = os.path.join(TEMP_DIR, protocol_name)
    os.makedirs(proto_temp_dir, exist_ok=True)

    OUTPUT_DIR = Path(OUTPUT_ROOT) / f"noddi_{protocol_name}_{JOB_ID}"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # ---- Extract volumes (contiguous block) --------------------------
    indices = list(range(start_vol, start_vol + n_vols))
    print(f"  Extracting {n_vols} volumes (vols {start_vol}–{start_vol + n_vols - 1})...")
    extracted_data = full_dwi_data[..., indices]

    # Save extracted DWI to temp dir
    proto_dwi_path = os.path.join(proto_temp_dir, "data.nii.gz")
    nib.save(nib.Nifti1Image(extracted_data.astype(np.float32), full_affine, full_header),
             proto_dwi_path)
    print(f"  ✓ Saved extracted DWI: {extracted_data.shape}")

    # Copy mask
    proto_mask_path = os.path.join(proto_temp_dir, "nodif_brain_mask.nii.gz")
    shutil.copy2(MASK_FILE_PATH, proto_mask_path)

    # ---- Slice bvals/bvecs and create AMICO scheme -------------------
    sub_bvals = full_bvals[indices]             # (n_vols,)
    sub_bvecs = full_bvecs[:, indices]          # (3, n_vols)

    sub_bvals_rounded = np.array([round_to_shell(b) for b in sub_bvals])
    bval_rounded_path = os.path.join(proto_temp_dir, "bvals_rounded")
    bvec_temp_path    = os.path.join(proto_temp_dir, "bvecs")
    np.savetxt(bval_rounded_path, sub_bvals_rounded, fmt='%.1f', newline=' ')
    np.savetxt(bvec_temp_path,    sub_bvecs,         fmt='%.6f')

    scheme_file = os.path.join(proto_temp_dir, "NODDI.scheme")
    scheme = create_amico_scheme(bval_rounded_path, bvec_temp_path, scheme_file)
    print(f"  ✓ AMICO scheme created: {len(scheme)} volumes")

    # ---- AMICO fitting -----------------------------------------------
    print("  Setting up AMICO evaluation...")
    ae = amico.Evaluation(proto_temp_dir, ".")
    ae.load_data(
        dwi_filename    = "data.nii.gz",
        scheme_filename = "NODDI.scheme",
        mask_filename   = "nodif_brain_mask.nii.gz",
        b0_thr          = B0_THRESHOLD
    )
    ae.set_model("NODDI")

    print("  Generating kernels...")
    ae.generate_kernels(regenerate=True)
    ae.load_kernels()

    print("  Fitting NODDI model...")
    t_fit = time.time()
    ae.fit()
    elapsed = time.time() - t_fit
    print(f"  ✓ Fitting done in {elapsed/60:.1f} min")

    ae.save_results()

    # ---- Copy results to final output dir ----------------------------
    amico_results_dir = os.path.join(proto_temp_dir, "AMICO", "NODDI")
    noddi_files = ["fit_NDI.nii.gz", "fit_ODI.nii.gz", "fit_FWF.nii.gz",
                   "fit_dir.nii.gz", "config.pickle"]

    print(f"  Copying results to {OUTPUT_DIR}")
    for fname in noddi_files:
        src = os.path.join(amico_results_dir, fname)
        dst = OUTPUT_DIR / fname
        if os.path.exists(src):
            shutil.copy2(src, dst)
            print(f"    ✓ {fname} ({os.path.getsize(src)/1e6:.1f} MB)")
        else:
            print(f"    ✗ {fname} NOT FOUND")

    # Save protocol metadata
    with open(OUTPUT_DIR / "protocol_info.txt", "w") as f:
        f.write(f"protocol: {protocol_name}\nshell: {shell}\ndirs: {ndir}\n"
                f"start_vol: {start_vol}\nn_vols: {n_vols}\n"
                f"fit_time_min: {elapsed/60:.1f}\njob_id: {JOB_ID}\n")

    # ---- Cleanup temp dir to save disk space -------------------------
    shutil.rmtree(proto_temp_dir)
    print(f"  ✓ Temp dir removed")

    # ---- Statistics --------------------------------------------------
    ndi = nib.load(OUTPUT_DIR / "fit_NDI.nii.gz").get_fdata()
    odi = nib.load(OUTPUT_DIR / "fit_ODI.nii.gz").get_fdata()
    fwf = nib.load(OUTPUT_DIR / "fit_FWF.nii.gz").get_fdata()
    mask_data = nib.load(MASK_FILE_PATH).get_fdata() > 0

    stats = {}
    for name, data in [("NDI", ndi), ("ODI", odi), ("FWF", fwf)]:
        vals = data[mask_data]
        vals = vals[np.isfinite(vals) & (vals > 0)]
        stats[name] = {'mean': vals.mean(), 'std': vals.std()}

    all_results[protocol_name] = {
        'shell': shell, 'ndir': ndir,
        'n_volumes': n_vols,
        'fitting_time_min': elapsed / 60,
        'stats': stats
    }

    print(f"\n  {protocol_name} — NDI: {stats['NDI']['mean']:.4f} | "
          f"ODI: {stats['ODI']['mean']:.4f} | FWF: {stats['FWF']['mean']:.4f}")
    print(f"  ✓ {protocol_name} COMPLETE\n")

## 4. Summary of All Protocols

In [ ]:
print("\n" + "="*70)
print("NODDI LAB SUBSAMPLING STUDY COMPLETE")
print("="*70)
print(f"\nJob ID:    {JOB_ID}")
print(f"Protocols: {len(all_results)}/{len(protocols_to_run)}")

print(f"\n{'Protocol':<12s} {'Shell':>6s} {'Dirs':>5s} {'Vols':>5s} {'Time(m)':>8s} {'NDI':>8s} {'ODI':>8s} {'FWF':>8s}")
print("-" * 67)

for protocol_name in sorted(all_results.keys()):
    r = all_results[protocol_name]
    s = r['stats']
    print(f"{protocol_name:<12s} {r['shell']:6d} {r['ndir']:5d} {r['n_volumes']:5d} "
          f"{r['fitting_time_min']:8.2f} {s['NDI']['mean']:8.4f} {s['ODI']['mean']:8.4f} {s['FWF']['mean']:8.4f}")

print("="*70)
print(f"Output: {OUTPUT_ROOT}/noddi_{{protocol}}_{JOB_ID}/")
print("="*70)